<br><br><br><br>

# 🟩 DB 연동 클래스 체계를 Score문제에 적용해보기
- DB 연동 클래스 체계를 어제 풀었던 Score문제에 적용해보세요.

## 🟢 MySQL DB Table(tb_scroe in mydb) 만들기
- 이미 어제 만들어 놓았습니다.
- 아래 코드만 참고하도록 하겠습니다.

In [ ]:
CREATE TABLE tb_score(
					id bigint PRIMARY KEY AUTO_INCREMENT,
					sname varchar(20) NOT NULL,
					kor int NOT NULL,
					eng int NOT NULL,
					mat Int NOT NULL,
					regdate datetime
);

INSERT INTO tb_score(sname, kor,  eng, mat, regdate)
values('홍길동', 90,90,90, now());


SELECT 
					sname, kor, eng, mat, (kor+eng+mat) AS total, 
					date_format(regdate, '%Y-%m-%d %H:%i') regdate
FROM tb_score;

## 🟢 DB 연동 클래스

In [ ]:
import pymysql
import pymysql.cursors


class Database:
    def __init__(self):
        self.conn = self.mysql_conn()
        self.cursor = self.conn.cursor(pymysql.cursors.DictCursor)

    # init 해줄것을 이렇게 만들었습니다.
    def mysql_conn(self):
        conn = pymysql.connect(
            host="localhost",
            user="root",
            password="",
            db="mydb",
            port=3306,
        )
        print("접속 성공")
        return conn

    # execute = 실행하다.
    # insert, update, delete 할 때 사용할 수 있게 이렇게 만들었다.
    def execute(self, query, args=()):
        # args - tuple 기본값
        print(args)
        self.cursor.execute(query, args)
        self.conn.commit()

    # 데이터 딱 1개만 가져오기
    # scalar 쿼리 포함,  select count(*) from tb_member
    def executeOne(self, query, args=()):
        self.cursor.execute(query, args)
        row = self.cursor.fetchone()
        return row  # 결과를 반환해야하낟. 첫번째 레코드값 하나만 가져간다.

    # 데이터 여러개 가져오기
    def executeAll(self, query, args=()):
        self.cursor.execute(query, args)
        rows = self.cursor.fetchall()
        return rows

    # 닫기
    def close(self):
        if self.conn.open:
            self.conn.close

    

## 🟢 DB 연동 클래스 Score 문제에 적용


In [13]:

class Score:
    def __init__(self):
        self.db = Database()

    # 1. 내역 전체 보기
    def all_view_data(self):
        sql = """
            SELECT
                id, sname, kor, eng, mat, 
                (kor+eng+mat) AS total,
                (kor+eng+mat)/3 AS average
            FROM tb_score;
        """
        # print(sql)
        self.db.execute(sql)
        rows = self.db.cursor.fetchall()
        print("데이터 개수", len(rows))
        for row in rows:
            print(
                row["id"],
                row["sname"],
                row["kor"],
                row["eng"],
                row["mat"],
                row["total"],
                row["average"]
            )
        print()

    # sname vaildation
    def validate_name(self, sname):
        sql = """
            select count(*) cnt from tb_score where sname = %s
        """
        row = self.db.executeOne(sql, (sname))
        if row["cnt"] == 0:
            return False
        return True

    # 2. 
    def insert_data(self):
        sname = input("이름 : ")
        if self.validate_name(sname) == True:
            return print("🚫 이미 존재하는 이름입니다. 다시 실행해주세요.")
        else:
            kor = input("국어 : ")
            eng = input("영어 : ")
            mat = input("수학 : ")
            sql = """
                INSERT INTO tb_score(sname, kor,  eng, mat, regdate)
                values(%s, %s, %s, %s, now())
            """
            self.db.execute(sql, (sname, kor, eng, mat)) # cursor 객체는 내부적으로 상태(state)를 가지고 있다
            print("INSERT 완료")
            print("\n----------- 📝 전체 성적 내역 확인 -----------")
            self.all_view_data()
            print()

    def update_data(self):
        # 수정 전 전체 리스트 파악
        print("\n----------- ✅ 수정할 데이터를 선택하세요 -----------")
        self.all_view_data()
        # input으로 입력받기
        id = input("수정할 id 입력 : ")
        sname = input("이름 : ")
        kor = input("국어 : ")
        eng = input("영어 : ")
        mat = input("수학 : ")
        sql = """
            UPDATE tb_score
            SET 
                sname = %s,
                kor = %s,
                eng = %s,
                mat = %s
            WHERE id = %s
        """
        self.db.execute(sql, (sname, kor, eng, mat, id))
        print("수정 완료")
        print("\n----------- 📝 전체 성적 내역 확인 -----------")
        self.all_view_data()
        print()

    def delete_data(self):
        # 삭제 전 전체 리스트 파악
        print("----------- ✅ 삭제할 데이터를 선택하세요 -----------")
        self.all_view_data()
        # input으로 입력받기
        sname = input("삭제할 이름을 입력하세요 : ")
        sql = """
            DELETE FROM tb_score WHERE sname = %s
        """
        self.db.execute(sql, sname)
        print("삭제 완료")
        print("\n----------- 📝 전체 성적 내역 확인 -----------")
        self.all_view_data()
        print()

    def start(self):
        while True:
            print(f"1.전체보기  |  2.추가  |  3.수정  |  4.삭제  |  0.종료")
            select = input("🔢 번호 선택: ")

            if select == "1":
                self.all_view_data()
            elif select == "2":
                self.insert_data()
            elif select == "3":
                self.update_data()
            elif select == "4":
                self.delete_data()
            elif select == "0":
                break

if __name__ == "__main__":
    s = Score()
    s.start()


접속 성공
1.전체보기  |  2.추가  |  3.수정  |  4.삭제  |  0.종료
()
데이터 개수 4
1 홍길동 90 90 90 270 90.0000
2 김민지 55 78 97 230 76.6667
4 강낭콩 22 82 34 138 46.0000
6 피카츄 20 30 40 90 30.0000

1.전체보기  |  2.추가  |  3.수정  |  4.삭제  |  0.종료
🚫 이미 존재하는 이름입니다. 다시 실행해주세요.
1.전체보기  |  2.추가  |  3.수정  |  4.삭제  |  0.종료


### 🟡 같이 풀어보기

In [15]:
class ScoreData:
  def __init__(self, sname = "", kor=0, eng=0, mat=0, total=0, average=0):
    self.db = Database()
    self.sname = sname
    self.kor = kor
    self.eng = eng
    self.mat = mat
    self.total = total
    self.average = average

  def output(self):
    print(self.sname, self.kor, self.eng, self.mat, self.total, self.average)


class ScoreManager:
  def __init__(self):
    self.db = Database()

  def append(self):
    s = ScoreData()
    s.sname = input("이름 : ")
    s.kor = int(input("국어 : "))
    s.eng = int(input("영어 : "))
    s.mat = int(input("수학 : "))
    sql = """
      insert into tb_score(sname, kor, eng, mat, regdate)
      values( %s, %s, %s, %s, now())
    """
    self.db.execute(sql, (s.sname, s.kor, s.eng, s.mat))
    self.db.close()

  def output(self):
      sql = """
            SELECT
                sname, kor, eng, mat, 
                (kor+eng+mat) as toal,
                (kor+eng+mat)/3 as average
            FROM tb_score;
        """
      rows = self.db.executeAll(sql)
      print("데이터 개수", len(rows))
      self.dataList = []
      for r in rows:
          s = ScoreData(
            r['sname'], r['kor'], r['eng'], r['mat'], 
            r['total'], r['average']
          )
          self.dataList.append(s)

      for s in self.dataList:
        s.output()
      print()
    
if __name__ == "__main__":
    sm = ScoreManager()
    sm.append()

접속 성공
접속 성공
('테스트2', 10, 20, 30)


## 🟢🔥 마무리 공부하기

- 🟡 어려운 것들이 존재한다.
  - conn : MySQL 건물로 들어가는 문
  - cursor : 건물 안에서 명령을 수행하는 직원
  - execute() : 직원에게 시키는 업무
    - executemany() : 여러 개 반복 실행 / 리스트로 여러 데이터를 한꺼번에 insert 등등
  - fetchone() : 결과에서 1줄 가져오기
    - fetchall() : 결과에서 모든 줄 가져오기
  - args : 업무에 필요한 자료들

- 🟡 명령어를 이어서 생각해보자!!!
  - conn
    - conn   # db 연결
    - conn.commit()   # 변경사항적용
  - cursor   # db 연결되었으니, 조작해!
    - conn.cursor(pymysql.cursors.DictCursor)   # dict type으로 가져올게
    - 🔥 conn.cursor(~~).execute("SELECT * FROM tb_member WHERE user_id = %s", ("boram",))    # 실제 sql 쿼리를 실행
    - conn.cursor.executemany()
    - conn.cursor.fetchone()
    - conn.cursor.fetchall()